# Support Vector Machine (SVM) - Training Notebook

This notebook contains a simplified training script for the SVM classifier.
For comprehensive hyperparameter tuning and evaluation, see `SVM_Classifier.ipynb`.

## Features:
- Data loading and preprocessing
- Feature standardization
- Model training with configurable parameters
- Cross-validation analysis
- Model evaluation and saving

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import matplotlib.pyplot as plt
import seaborn as sns

from svm_model import SVMClassifier

# Set random seeds for reproducibility
np.random.seed(42)

print("✓ All libraries imported successfully!")

## 2. Configuration and Data Loading

In [ ]:
# Configure file paths
DATA_DIR = "./data"
MODELS_DIR = "./models"
RESULTS_DIR = "./results"

# Create directories
for directory in [MODELS_DIR, RESULTS_DIR]:
    os.makedirs(directory, exist_ok=True)

# Data file paths
pkl_path = os.path.join(DATA_DIR, 'training_data.pkl')
csv_path = os.path.join(DATA_DIR, 'training_data.csv')

print(f"Data directory: {DATA_DIR}")
print(f"Models directory: {MODELS_DIR}")
print(f"Results directory: {RESULTS_DIR}")

In [ ]:
# Load the dataset
if os.path.exists(pkl_path):
    with open(pkl_path, 'rb') as f:
        df = pickle.load(f)
    print(f"✓ Loaded dataset from pickle file")
elif os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"✓ Loaded dataset from CSV file")
else:
    raise FileNotFoundError(f"Data file not found at {pkl_path} or {csv_path}")

print(f"\nDataset Shape: {df.shape}")
print(f"Features: {df.shape[1] - 1}")
print(f"Samples: {df.shape[0]}")
print(f"\nFirst few rows:")
print(df.head())

## 3. Data Preprocessing

In [ ]:
# Separate features and labels
X = df.drop('label', axis=1).values
y = df['label'].values

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Unique classes: {np.unique(y)}")

# Encode labels if necessary
if y.dtype == 'object':
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y)
    print(f"\nClass encoding: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")
else:
    label_encoder = None
    print("\nLabels are already numeric")

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"\nClass distribution (Training):")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Class {u}: {c} samples ({100*c/len(y_train):.1f}%)")

## 4. Model Training

In [ ]:
# Create and train SVM model
num_features = X_train.shape[1]
num_classes = len(np.unique(y))

print(f"Creating SVM model...")
print(f"  Features: {num_features}")
print(f"  Classes: {num_classes}")

# Initialize SVM model with default parameters
model = SVMClassifier(
    num_features=num_features,
    num_classes=num_classes,
    kernel='rbf',
    C=1.0,
    gamma='scale'
)

print("\nTraining SVM model...")
model.fit(X_train, y_train)
print("✓ Model training completed!")

## 5. Model Evaluation

In [ ]:
# Make predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Calculate metrics
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

train_f1 = f1_score(y_train, y_train_pred, average='weighted')
test_f1 = f1_score(y_test, y_test_pred, average='weighted')

print("="*60)
print("MODEL PERFORMANCE")
print("="*60)
print("\nTraining Set:")
print(f"  Accuracy: {train_accuracy:.4f}")
print(f"  F1-Score: {train_f1:.4f}")

print("\nTesting Set:")
print(f"  Accuracy: {test_accuracy:.4f}")
print(f"  F1-Score: {test_f1:.4f}")

print(f"\nOverfitting Check:")
acc_diff = train_accuracy - test_accuracy
print(f"  Accuracy difference: {acc_diff:.4f}")

In [ ]:
# Classification report
print("\n" + "="*60)
print("CLASSIFICATION REPORT (Testing Set)")
print("="*60)
print(classification_report(y_test, y_test_pred))

## 6. Cross-Validation

In [ ]:
# Perform k-fold cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_fold_train = X_train[train_idx]
    y_fold_train = y_train[train_idx]
    X_fold_val = X_train[val_idx]
    y_fold_val = y_train[val_idx]
    
    fold_model = SVMClassifier(num_features, num_classes)
    fold_model.fit(X_fold_train, y_fold_train)
    
    y_fold_pred = fold_model.predict(X_fold_val)
    fold_accuracy = accuracy_score(y_fold_val, y_fold_pred)
    cv_scores.append(fold_accuracy)
    
    print(f"Fold {fold}: {fold_accuracy:.4f}")

print(f"\nCross-Validation Results:")
print(f"  Mean Accuracy: {np.mean(cv_scores):.4f}")
print(f"  Std Dev: {np.std(cv_scores):.4f}")

## 7. Save Model

In [ ]:
# Save the trained model
model_path = os.path.join(MODELS_DIR, 'svm_trained.pkl')
model.save(model_path)
print(f"✓ Model saved to {model_path}")